In [ ]:
#first we take in raw trip journeys from TFL and then we aggregate them by day, route, and stop. 
# We also calculate the total number of journeys for each group. 
# Finally, we save the aggregated data to a new CSV file.

In [9]:
from pathlib import Path
import pandas as pd
from typing import Literal

AggSide = Literal["start", "end", "both"]
Freq = Literal["h", "d"]


START_ID_NORMS = [
    "startstationid",
    "startstationlogicalterminal",
    "startstationnumber",
    "startstationid",
]
START_DATE_NORMS = [
    "startdate",
    "startdatetime",
    "starttime",
]
END_ID_NORMS = [
    "endstationid",
    "endstationlogicalterminal",
    "endstationnumber",
]
END_DATE_NORMS = [
    "enddate",
    "enddatetime",
    "endtime",
]



def _norm(s: str) -> str:
    # lower-case and remove spaces/underscores for robust matching
    return "".join(ch for ch in s.lower() if ch.isalnum())

def _pick_col_norm(cols: list[str], candidate_norms: list[str]) -> str | None:
    norm_to_real = {_norm(c): c for c in cols}
    for cn in candidate_norms:
        if cn in norm_to_real:
            return norm_to_real[cn]
    return None



def aggregate_from_folder_chunked(
    folder_path: str | Path,
    *,
    freq: Freq = "h",
    side: AggSide = "start",
    chunksize: int = 200_000,
    dayfirst: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    folder = Path(folder_path)
    files = sorted(folder.rglob("*.csv"))
    if not files:
        raise ValueError("No CSV files found in folder.")
    if verbose:
        print(f"Found {len(files)} files.")

    acc = None  # keep accumulator as None until first successful file

    skipped = []

    for f in files:
        if verbose:
            print(f"Processing {f.name} ...")

        # Read header only to detect schema
        try:
            header = pd.read_csv(f, nrows=0)
        except Exception as e:
            skipped.append((f.name, f"header_read_failed: {e}"))
            continue

        cols = list(header.columns)

        start_id_col   = _pick_col_norm(cols, START_ID_NORMS)
        start_date_col = _pick_col_norm(cols, START_DATE_NORMS)
        end_id_col     = _pick_col_norm(cols, END_ID_NORMS)
        end_date_col   = _pick_col_norm(cols, END_DATE_NORMS)

        needed = []
        if side in ("start", "both"):
            if start_id_col is None or start_date_col is None:
                skipped.append((f.name, "missing start columns"))
                continue
            needed += [start_id_col, start_date_col]
        if side in ("end", "both"):
            if end_id_col is None or end_date_col is None:
                skipped.append((f.name, "missing end columns"))
                continue
            needed += [end_id_col, end_date_col]

        # Deduplicate
        needed = list(dict.fromkeys(needed))

        # Process in chunks
        try:
            for chunk in pd.read_csv(
                f,
                usecols=needed,
                chunksize=chunksize,
                low_memory=True,
            ):
                parts = []

                if side in ("start", "both"):
                    tmp = chunk[[start_id_col, start_date_col]].copy()
                    tmp = tmp.dropna()
                    tmp["station_id"] = pd.to_numeric(tmp[start_id_col], errors="coerce")
                    tmp["ts"] = pd.to_datetime(tmp[start_date_col], dayfirst=dayfirst, errors="coerce")
                    tmp = tmp.dropna(subset=["station_id", "ts"])
                    tmp["station_id"] = tmp["station_id"].astype(int)
                    tmp["ts"] = tmp["ts"].dt.floor(freq)
                    g = tmp.groupby(["station_id", "ts"], as_index=False).size()
                    g = g.rename(columns={"size": "trips_start"})
                    parts.append(g)

                if side in ("end", "both"):
                    tmp = chunk[[end_id_col, end_date_col]].copy()
                    tmp = tmp.dropna()
                    tmp["station_id"] = pd.to_numeric(tmp[end_id_col], errors="coerce")
                    tmp["ts"] = pd.to_datetime(tmp[end_date_col], dayfirst=dayfirst, errors="coerce")
                    tmp = tmp.dropna(subset=["station_id", "ts"])
                    tmp["station_id"] = tmp["station_id"].astype(int)
                    tmp["ts"] = tmp["ts"].dt.floor(freq)
                    g = tmp.groupby(["station_id", "ts"], as_index=False).size()
                    g = g.rename(columns={"size": "trips_end"})
                    parts.append(g)

                if not parts:
                    continue

                chunk_agg = parts[0]
                for p in parts[1:]:
                    chunk_agg = chunk_agg.merge(p, on=["station_id", "ts"], how="outer")

                if acc is None:
                    acc = chunk_agg
                else:
                    acc = pd.concat([acc, chunk_agg], ignore_index=True)
                    # sum within station-time
                    sum_cols = [c for c in ["trips_start", "trips_end"] if c in acc.columns]
                    acc = acc.groupby(["station_id", "ts"], as_index=False)[sum_cols].sum()

        except Exception as e:
            skipped.append((f.name, f"read_or_parse_failed: {e}"))
            continue

    if acc is None:
        raise RuntimeError("No files were successfully processed. Check skipped log.")

    # Ensure missing trip cols exist
    if "trips_start" not in acc.columns:
        acc["trips_start"] = 0
    if "trips_end" not in acc.columns:
        acc["trips_end"] = 0

    acc = acc.sort_values(["station_id", "ts"]).reset_index(drop=True)

    if verbose and skipped:
        print("\nSkipped files summary (first 20):")
        for name, reason in skipped[:20]:
            print(f"  - {name}: {reason}")
        if len(skipped) > 20:
            print(f"  ... and {len(skipped)-20} more")

    return acc





In [11]:
base = aggregate_from_folder_chunked(
    r"E:\tfl_project\data",
    freq="h",
    side="start",
    chunksize=100_000,   # safer if you’re memory-tight
)


Found 117 files.
Processing 01aJourneyDataExtract10Jan16-23Jan16.csv ...
Processing 01bJourneyDataExtract24Jan16-06Feb16.csv ...
Processing 02aJourneyDataExtract07Fe16-20Feb2016.csv ...
Processing 02bJourneyDataExtract21Feb16-05Mar2016.csv ...
Processing 03JourneyDataExtract06Mar2016-31Mar2016.csv ...
Processing 04JourneyDataExtract01Apr2016-30Apr2016.csv ...
Processing 05JourneyDataExtract01May2016-17May2016.csv ...
Processing 06JourneyDataExtract18May2016-24May2016.csv ...
Processing 07JourneyDataExtract25May2016-31May2016.csv ...
Processing 08JourneyDataExtract01Jun2016-07Jun2016.csv ...
Processing 09JourneyDataExtract08Jun2016-14Jun2016.csv ...
Processing 100JourneyDataExtract07Mar2018-13Mar2018.csv ...
Processing 101JourneyDataExtract14Mar2018-20Mar2018.csv ...
Processing 102JourneyDataExtract21Mar2018-27Mar2018.csv ...
Processing 103JourneyDataExtract28Mar2018-03Apr2018.csv ...
Processing 104JourneyDataExtract04Apr2018-10Apr2018.csv ...
Processing 105JourneyDataExtract11Apr2018-1

In [12]:
base.head()

,station_id,ts,trips_start,trips_end
0,1,2016-01-10 09:00:00,4,0
1,1,2016-01-10 10:00:00,1,0
2,1,2016-01-10 11:00:00,2,0
3,1,2016-01-10 12:00:00,2,0
4,1,2016-01-10 13:00:00,2,0


In [14]:
#save aggregated basefile
base.to_csv(r"E:\tfl_project\agg_basefile.csv", index=False)

In [15]:
base.shape

(7965747, 4)

In [5]:
test.head()

,Rental Id,Duration,Bike Id,End Date,EndStation Id,EndStation Name,Start Date,StartStation Id,StartStation Name
0,50754225,240,11834,10/01/2016 00:04,383.0,"Frith Street, Soho",10/01/2016 00:00,18,"Drury Lane, Covent Garden"
1,50754226,300,9648,10/01/2016 00:05,719.0,"Victoria Park Road, Hackney Central",10/01/2016 00:00,479,"Pott Street, Bethnal Green"
2,50754227,1200,10689,10/01/2016 00:20,272.0,"Baylis Road, Waterloo",10/01/2016 00:00,425,"Harrington Square 2, Camden Town"
3,50754228,780,8593,10/01/2016 00:14,471.0,"Hewison Street, Old Ford",10/01/2016 00:01,487,"Canton Street, Poplar"
4,50754229,600,8619,10/01/2016 00:11,399.0,"Brick Lane Market, Shoreditch",10/01/2016 00:01,501,"Cephas Street, Bethnal Green"


In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

csv_file = 'E:\\tfl_project\\agg_basefile.csv'
parquet_file = 'E:\\tfl_project\\agg_basefile_pq.parquet'
chunksize = 100_000

csv_stream = pd.read_csv(csv_file, chunksize=chunksize, low_memory=False)

for i, chunk in enumerate(csv_stream):
    print("Chunk", i)
    if i == 0:
        # Guess the schema of the CSV file from the first chunk
        parquet_schema = pa.Table.from_pandas(df=chunk).schema
        # Open a Parquet file for writing
        parquet_writer = pq.ParquetWriter(parquet_file, parquet_schema, compression='snappy')
    # Write CSV chunk to the parquet file
    table = pa.Table.from_pandas(chunk, schema=parquet_schema)
    parquet_writer.write_table(table)

parquet_writer.close()

Chunk 0
Chunk 1
Chunk 2
Chunk 3
Chunk 4
Chunk 5
Chunk 6
Chunk 7
Chunk 8
Chunk 9
Chunk 10
Chunk 11
Chunk 12
Chunk 13
Chunk 14
Chunk 15
Chunk 16
Chunk 17
Chunk 18
Chunk 19
Chunk 20
Chunk 21
Chunk 22
Chunk 23
Chunk 24
Chunk 25
Chunk 26
Chunk 27
Chunk 28
Chunk 29
Chunk 30
Chunk 31
Chunk 32
Chunk 33
Chunk 34
Chunk 35
Chunk 36
Chunk 37
Chunk 38
Chunk 39
Chunk 40
Chunk 41
Chunk 42
Chunk 43
Chunk 44
Chunk 45
Chunk 46
Chunk 47
Chunk 48
Chunk 49
Chunk 50
Chunk 51
Chunk 52
Chunk 53
Chunk 54
Chunk 55
Chunk 56
Chunk 57
Chunk 58
Chunk 59
Chunk 60
Chunk 61
Chunk 62
Chunk 63
Chunk 64
Chunk 65
Chunk 66
Chunk 67
Chunk 68
Chunk 69
Chunk 70
Chunk 71
Chunk 72
Chunk 73
Chunk 74
Chunk 75
Chunk 76
Chunk 77
Chunk 78
Chunk 79


: 

In [2]:
import pandas as pd
base = pd.read_csv(r"E:\tfl_project\agg_basefile.csv")

In [3]:
base.head()

,station_id,ts,trips_start,trips_end
0,1,2016-01-10 09:00:00,4,0
1,1,2016-01-10 10:00:00,1,0
2,1,2016-01-10 11:00:00,2,0
3,1,2016-01-10 12:00:00,2,0
4,1,2016-01-10 13:00:00,2,0


In [4]:
import pandas as pd
import matplotlib.pyplot as plt

# base must have: station_id, ts, trips_start (and optionally strike_exposed)
base = base.copy()
base["ts"] = pd.to_datetime(base["ts"])
base["date"] = base["ts"].dt.floor("d")
base["hour"] = base["ts"].dt.hour
base["dow"] = base["ts"].dt.dayofweek
base["is_weekend"] = (base["dow"] >= 5).astype(int)

C:\Users\lukes\AppData\Local\Temp\ipykernel_16472\2143514955.py:7: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  base["date"] = base["ts"].dt.floor("d")


In [ ]:
#let's also join on strike data. We need to know which lines connect to which station IDs 

In [12]:
import json
import pandas as pd

def load_bike_station_locations(filepath: str) -> pd.DataFrame:
    """
    Returns DataFrame with:
      station_id, lat, lon, name
    """
    with open(filepath, encoding="utf-8") as f:
        stations = json.load(f)

    rows = []
    for station in stations:
        #station = int(station)
        station_id = station["id"].split("_")[1]
        rows.append({
            "station_id": int(station_id),
            "lat": station["lat"],
            "lon": station["lon"],
            "station_name": station["commonName"]
        })

    return pd.DataFrame(rows)

In [13]:
bike_stations = load_bike_station_locations("E:\\tfl_project\data\BikePoint.json")

In [25]:
bike_stations

,station_id,lat,lon,station_name
0,1,51.529163,-0.109970,"River Street , Clerkenwell"
1,2,51.499606,-0.197574,"Phillimore Gardens, Kensington"
2,3,51.521283,-0.084605,"Christopher Street, Liverpool Street"
3,4,51.530059,-0.120973,"St. Chad's Street, King's Cross"
4,5,51.493130,-0.156876,"Sedding Street, Sloane Square"
...,...,...,...,...
795,883,51.541190,-0.058826,"London Fields, Hackney Central"
796,884,51.509158,-0.224103,"Westfield Ariel Way, White City"
797,885,51.537349,-0.147154,"Gloucester Avenue, Camden Town"
798,887,51.536922,-0.150181,"London Zoo Car Park, The Regent's Park"


In [29]:
import requests
import pandas as pd

def fetch_tube_stations_and_lines() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      tube_stations: tube_station_id, tube_station_name, lat, lon
      tube_lines: tube_station_id, affected_line
    """
    url = "https://api.tfl.gov.uk/StopPoint/Mode/tube"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    data = r.json()

    # The response is usually a dict with 'stopPoints'
    stop_points = data.get("stopPoints", data)

    st_rows = []
    line_rows = []

    UNDERGROUND_LINES = {
    "bakerloo",
    "central",
    "circle",
    "district",
    "hammersmith-city",
    "jubilee",
    "metropolitan",
    "northern",
    "piccadilly",
    "victoria",
    "waterloo-city"
    }   

    for sp in stop_points:
        tube_station_id = sp.get("naptanId") or sp.get("id")
        name = sp.get("commonName")
        lat = sp.get("lat")
        lon = sp.get("lon")

        if tube_station_id is None or lat is None or lon is None:
            continue

        st_rows.append({
            "tube_station_id": tube_station_id,
            "tube_station_name": name,
            "lat": lat,
            "lon": lon
        })

        # Lines served can appear under 'lines'
        for ln in sp.get("lines", []) or []:
            # lineId values are like 'central', 'piccadilly', etc.
            line_id = ln.get("id")
            if line_id in UNDERGROUND_LINES:
                line_rows.append({
                    "tube_station_id": tube_station_id,
                    "affected_line": f"{line_id}_line"
                })

    tube_stations = pd.DataFrame(st_rows).drop_duplicates(subset=["tube_station_id"])
    tube_lines = pd.DataFrame(line_rows).drop_duplicates()

    return tube_stations, tube_lines


tube_stations, tube_lines = fetch_tube_stations_and_lines()
tube_stations.head(), tube_lines.head()

(  tube_station_id                       tube_station_name        lat       lon
 0    0400ZZLUAMS0            Amersham Underground Station  51.674206 -0.607362
 1    0400ZZLUCAL0  Chalfont & Latimer Underground Station  51.667915 -0.560616
 2    0400ZZLUCAL1  Chalfont & Latimer Underground Station  51.668122 -0.560624
 3    0400ZZLUCSM0             Chesham Underground Station  51.705227 -0.611113
 4    2100ZZLUCXY0             Croxley Underground Station  51.647069 -0.441746,
   tube_station_id    affected_line
 0    9400ZZBPSUST    northern_line
 1    9400ZZLUACT1    district_line
 2    9400ZZLUACT2    district_line
 3    9400ZZLUACT3  piccadilly_line
 4    9400ZZLUACT4  piccadilly_line)

In [20]:
tube_lines['affected_line'].value_counts()

affected_line
district_line            192
northern_line            166
piccadilly_line          162
central_line             154
circle_line              118
metropolitan_line        112
hammersmith-city_line     95
jubilee_line              90
bakerloo_line             90
victoria_line             57
waterloo-city_line         8
Name: count, dtype: int64

In [42]:
strike_data = pd.read_csv(r"E:\tfl_project\strikes\strikes.csv")
strike_data

,date_start,date_end,affected_line
0,07/11/18,07/11/18,central_line
1,05/10/18,05/10/18,central_line
2,26/09/18,29/09/18,piccadilly_line
3,22/08/18,24/08/18,central_line
4,12/07/18,13/07/18,central_line
5,13/04/18,13/04/18,central_line
6,07/05/17,08/05/17,northern_line
7,07/05/17,08/05/17,jubilee_line
8,21/02/17,22/02/17,central_line
9,08/01/17,09/01/17,all


In [21]:
import numpy as np

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def build_station_line_map(
    bike_stations: pd.DataFrame,   # station_id, lat, lon
    tube_stations: pd.DataFrame,   # tube_station_id, lat, lon
    tube_lines: pd.DataFrame,      # tube_station_id, affected_line
    radius_m: float = 800.0,
    fallback_to_nearest: bool = True,
) -> pd.DataFrame:
    """
    Returns station_line_map:
      station_id, affected_line, tube_station_id, dist_m
    """
    b = bike_stations[["station_id", "lat", "lon"]].dropna().copy()
    t = tube_stations[["tube_station_id", "lat", "lon"]].dropna().copy()

    b_lat = b["lat"].to_numpy()
    b_lon = b["lon"].to_numpy()
    t_lat = t["lat"].to_numpy()
    t_lon = t["lon"].to_numpy()
    t_id  = t["tube_station_id"].to_numpy()

    rows = []
    for i in range(len(b)):
        d = haversine_m(b_lat[i], b_lon[i], t_lat, t_lon)

        idx = np.where(d <= radius_m)[0]
        if len(idx) == 0 and fallback_to_nearest:
            idx = np.array([int(np.argmin(d))])

        for j in idx:
            rows.append({
                "station_id": int(b.iloc[i]["station_id"]),
                "tube_station_id": t_id[j],
                "dist_m": float(d[j])
            })

    station_tube = pd.DataFrame(rows).drop_duplicates()

    station_line_map = (
        station_tube.merge(tube_lines, on="tube_station_id", how="left")
        .dropna(subset=["affected_line"])
        [["station_id", "affected_line", "tube_station_id", "dist_m"]]
        .drop_duplicates()
    )
    return station_line_map


# bike_stations should already exist from your BikePoint JSON step
station_line_map = build_station_line_map(
    bike_stations=bike_stations,
    tube_stations=tube_stations,
    tube_lines=tube_lines,
    radius_m=800,
    fallback_to_nearest=True
)
station_line_map.head()

,station_id,affected_line,tube_station_id,dist_m
3,1,northern_line,9400ZZLUAGL1,409.921278
4,1,northern_line,9400ZZLUAGL2,404.627788
5,1,northern_line,940GZZLUAGL,404.627788
8,2,circle_line,9400ZZLUHSK1,383.443390
9,2,district_line,9400ZZLUHSK1,383.443390


In [22]:
station_line_map = station_line_map.drop_duplicates(
    subset=["station_id", "affected_line"]
).reset_index(drop=True)

station_line_map.head()

,station_id,affected_line,tube_station_id,dist_m
0,1,northern_line,9400ZZLUAGL1,409.921278
1,2,circle_line,9400ZZLUHSK1,383.443390
2,2,district_line,9400ZZLUHSK1,383.443390
3,3,central_line,9400ZZLULVT1,392.274839
4,3,circle_line,9400ZZLULVT3,472.502939


In [43]:
station_line_map.groupby("station_id")["affected_line"].nunique().describe()

count    719.000000
mean       3.141864
std        1.938300
min        1.000000
25%        1.000000
50%        3.000000
75%        5.000000
max        8.000000
Name: affected_line, dtype: float64

In [44]:
sorted(station_line_map["affected_line"].unique())[:50]
len(station_line_map["affected_line"].unique())

11

In [45]:
def build_station_day_treatment(strikes_daily, station_line_map):
    # Merge line-level strikes to station-level exposure
    treated = (
        strikes_daily
        .merge(station_line_map[["station_id", "affected_line"]],
               on="affected_line",
               how="inner")
        .drop_duplicates(subset=["station_id", "date"])
        .assign(strike_exposed=1)
        [["station_id", "date", "strike_exposed"]]
    )
    return treated

In [46]:
station_day_treat = build_station_day_treatment(strikes_daily, station_line_map)

base["date"] = base["ts"].dt.floor("D")
base = base.merge(station_day_treat,
                  on=["station_id", "date"],
                  how="left")

base["strike_exposed"] = base["strike_exposed"].fillna(0).astype(int)

: 

In [37]:
def expand_strikes_daily(strike_data: pd.DataFrame) -> pd.DataFrame:
    s = strike_data.copy()
    s["date_start"] = pd.to_datetime(s["date_start"], dayfirst=True, errors="coerce")
    s["date_end"] = pd.to_datetime(s["date_end"], dayfirst=True, errors="coerce")
    s = s.dropna(subset=["date_start", "date_end", "affected_line"]).copy()

    rows = []
    for r in s.itertuples(index=False):
        for d in pd.date_range(r.date_start.floor("D"), r.date_end.floor("D"), freq="D"):
            rows.append({"date": d, "affected_line": r.affected_line, "strike": 1})
    return pd.DataFrame(rows).drop_duplicates()

strikes_daily = expand_strikes_daily(strike_data)

C:\Users\lukes\AppData\Local\Temp\ipykernel_10360\2308408687.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s["date_start"] = pd.to_datetime(s["date_start"], dayfirst=True, errors="coerce")
C:\Users\lukes\AppData\Local\Temp\ipykernel_10360\2308408687.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s["date_end"] = pd.to_datetime(s["date_end"], dayfirst=True, errors="coerce")


In [40]:
strikes_daily

,date,affected_line,strike
0,2018-11-07,central_line,1
1,2018-10-05,central_line,1
2,2018-09-26,piccadilly_line,1
3,2018-09-27,piccadilly_line,1
4,2018-09-28,piccadilly_line,1
5,2018-09-29,piccadilly_line,1
6,2018-08-22,central_line,1
7,2018-08-23,central_line,1
8,2018-08-24,central_line,1
9,2018-07-12,central_line,1


In [41]:
new_lines = []

UNDERGROUND_LINES = [
    "bakerloo",
    "central",
    "circle",
    "district",
    "hammersmith-city",
    "jubilee",
    "metropolitan",
    "northern",
    "piccadilly",
    "victoria",
    "waterloo-city"
    ] 

for line in UNDERGROUND_LINES:
    new_dict_date_1 = {'date':'2017-01-08', 'affected_line': f'{line}_line', 'strike': 1}
    new_dict_date_2 = {'date':'2017-01-09', 'affected_line': f'{line}_line', 'strike': 1}
    print(f'adding strike entries for {line} line on 2017-01-08 and 2017-01-09')
    new_lines.append(new_dict_date_1)
    new_lines.append(new_dict_date_2)

strikes_daily.append(new_lines)
strikes_daily



adding strike entries for bakerloo line on 2017-01-08 and 2017-01-09
adding strike entries for central line on 2017-01-08 and 2017-01-09
adding strike entries for circle line on 2017-01-08 and 2017-01-09
adding strike entries for district line on 2017-01-08 and 2017-01-09
adding strike entries for hammersmith-city line on 2017-01-08 and 2017-01-09
adding strike entries for jubilee line on 2017-01-08 and 2017-01-09
adding strike entries for metropolitan line on 2017-01-08 and 2017-01-09
adding strike entries for northern line on 2017-01-08 and 2017-01-09
adding strike entries for piccadilly line on 2017-01-08 and 2017-01-09
adding strike entries for victoria line on 2017-01-08 and 2017-01-09
adding strike entries for waterloo-city line on 2017-01-08 and 2017-01-09


AttributeError: 'DataFrame' object has no attribute 'append'

In [36]:
strikes_daily

,date,affected_line,strike
0,2018-11-07,central_line,1
1,2018-10-05,central_line,1
2,2018-09-26,piccadilly_line,1
3,2018-09-27,piccadilly_line,1
4,2018-09-28,piccadilly_line,1
5,2018-09-29,piccadilly_line,1
6,2018-08-22,central_line,1
7,2018-08-23,central_line,1
8,2018-08-24,central_line,1
9,2018-07-12,central_line,1
